# Task 3 — How does an AI actually write?

In the last notebook we built classifiers that *sort* text. But how does something like ChatGPT
*write* text? In this notebook we'll build a tiny version of the idea behind it — and it uses only
Python's built-in tools, so there's **nothing to install**.

We'll do it in two parts:

1. **Tokenising** — how a computer chops text up and turns it into numbers.
2. **A next-word predictor** — a mini "language model" that learns from text and then writes its own.

This is the same core idea as the huge models behind ChatGPT: *predict the next word, over and over.*
Ours is tiny, so its writing is funny and clumsy — but the principle is identical.

---

### How to use this notebook
- Run each grey **code cell** by clicking it and pressing **▶** (or `Shift + Enter`).
- Run the cells **in order, top to bottom.**
- Things you can change are marked with `# 👉 CHANGE THIS`.
- If you see a red error, re-run the cell above it, then try again.

# Part 1 — Turning text into numbers

A computer can't work with words directly — it works with numbers. The first job of any language
model is to chop text into pieces (**tokens**) and give each one a number.

Run this to build a simple tokeniser.

In [ ]:
import re

def tokenise(text):
    """Split text into words and punctuation, all lowercase."""
    return re.findall(r"[a-z']+|[.,!?;]", text.lower())

sentence = "The cat sat on the mat!"   # 👉 CHANGE THIS
print(tokenise(sentence))

### Giving each word a number

Now we build a **vocabulary**: every different word gets its own number. Notice that the *same*
word always gets the *same* number — that's how the computer keeps track of it.

In [ ]:
def build_vocabulary(tokens):
    """Give each unique token its own id number."""
    vocab = {}
    for t in tokens:
        if t not in vocab:
            vocab[t] = len(vocab)
    return vocab

tokens = tokenise("the cat sat on the mat. the cat ran.")
vocab = build_vocabulary(tokens)

print("Vocabulary (word -> number):")
print(vocab)
print()
print("The sentence as numbers:")
print([vocab[t] for t in tokens])

Notice that **"the"** and **"cat"** each appear twice, and they get the same number each time.
The computer has turned language into a list of numbers it can actually work with.

### What about words it doesn't know?

Real models hit a problem: there are too many possible words, and people invent new ones. So instead
of storing every whole word, they break unfamiliar words into smaller **chunks**. Here's the idea
using single letters — a bit like sounding out a word you've never seen.

In [ ]:
word = "biology"   # 👉 CHANGE THIS  (try a long or made-up word!)

# Break the word into single-letter chunks
chunks = list(word)
print("Chunks:", chunks)

# Give each chunk a number
chunk_vocab = build_vocabulary(chunks)
print("As numbers:", [chunk_vocab[c] for c in chunks])

Real models like ChatGPT use chunks somewhere between single letters and whole words (called
**sub-words**) — so common words stay whole, but rare words get broken into familiar pieces.
This is why you sometimes see ChatGPT spell things out oddly!

# Part 2 — Predicting the next word

Remember the *"finish the sentence"* game? Given *"I left my umbrella at ___"*, you can guess
likely next words. A language model does exactly this: it learns, from lots of text, which words
tend to follow which.

We'll build one now. First, some text for it to learn from.

In [ ]:
# The text our model will learn from. 👉 CHANGE THIS — paste in song lyrics,
# a story, a Wikipedia paragraph, or your own writing! (More text = better results.)
training_text = """
The sun rose over the quiet village. The baker opened her shop and the smell of
fresh bread filled the street. Children ran to school while the old clock tower
rang out the hour. In the market, traders called out their prices and the crowd
moved slowly between the stalls. A small dog chased a cat across the square, and
everyone laughed. By the afternoon the sky turned grey and a gentle rain began
to fall. People hurried home, umbrellas open, and the village grew quiet again.
The baker closed her shop, the clock tower rang once more, and night fell over
the rooftops. The sun would rise again tomorrow, and the village would wake, and
the baker would open her shop, and the children would run to school once more.
"""

print(f"Loaded {len(training_text.split())} words of training text.")

### Teaching the model who-follows-who

We go through the text and, for every word, note which word comes **next**. The model ends up
knowing things like: after *"the"* you often see *"village"*, *"baker"*, or *"sun"*.

Run this to train it.

In [ ]:
from collections import defaultdict, Counter

def train(text, context=1):
    """Learn which token(s) tend to follow each sequence of `context` tokens."""
    tokens = tokenise(text)                   # reuse the tokeniser from Part 1
    model = defaultdict(Counter)
    for i in range(len(tokens) - context):
        key = tuple(tokens[i:i + context])   # the recent word(s)
        next_token = tokens[i + context]     # what came next
        model[key][next_token] += 1          # count it
    return model

# Train a model that looks at the 1 previous word.
model = train(training_text, context=1)

# Peek at what it learned: what tends to follow the word "the"?
print('Words that followed "the":')
print(dict(model[("the",)]))

That `Counter` shows the words that came after *"the"*, and how often each one did. The model
will use these counts as **probabilities** — words that appeared more often are more likely to be
picked.

### Let it write

Now the fun part. Starting from a word, the model repeatedly:
1. looks at the recent word(s),
2. picks a likely next word (weighted by the counts it learned),
3. adds it on — and repeats.

Run this a few times — you'll get different text each time!

In [ ]:
import random

def generate(model, context=1, length=40, start=None):
    """Generate text by repeatedly predicting the next word."""
    keys = list(model.keys())
    current = start if start else random.choice(keys)
    output = list(current)

    for _ in range(length):
        key = tuple(output[-context:])
        if key not in model:                 # dead end — jump somewhere random
            key = random.choice(keys)
        choices = model[key]
        words = list(choices.keys())
        weights = list(choices.values())     # more common words are more likely
        output.append(random.choices(words, weights=weights)[0])

    # join words back into text, tidying spaces before punctuation
    result = ""
    for w in output:
        result += w if w in ".,!?;" else ((" " if result else "") + w)
    return result

print(generate(model, context=1, length=40, start=("the",)))

It's clumsy and a bit nonsensical — but notice it's **not random**. It has learned the *style*
and common word-pairings of the text. This is exactly what ChatGPT does, only ChatGPT learned from
almost the entire internet and looks at hundreds of previous words, not just one.

### Experiment: how much "memory" should it have?

Our model above looked at just **1** previous word. What if it looks at **2**? It should sound more
coherent — but because our text is tiny, it also starts to just copy the original. Try it.

In [ ]:
context_size = 2   # 👉 CHANGE THIS — try 1, 2, or 3

model_n = train(training_text, context=context_size)
start = list(model_n.keys())[0]   # begin from the first key it learned
print(f"Looking at {context_size} previous word(s):\n")
print(generate(model_n, context=context_size, length=40, start=start))

**What you should notice:**
- **More context (2–3 words)** → more coherent, but with our tiny text it soon just *memorises* and
  repeats the original.
- **Less context (1 word)** → more varied and surprising, but more nonsense.

Real models solve this by training on *enormous* amounts of text, so they can use lots of context
**and** still say new things. That's a big part of why bigger models are better — and why they need
so much data and computing power.

### Your turn 🏆

1. Go back to the **training text** cell, paste in something you like (song lyrics, a football
   report, a fairy tale), and re-run the cells. What does the model's "voice" sound like now?
2. Find the `context_size` that gives the funniest results.
3. **Challenge:** the model can only ever reuse words from its training text. Why do you think that
   is — and what would you need to give it to fix that?

---
### 🎉 What you built

You built a **language model** — the same core idea behind ChatGPT — from scratch, using only
plain Python:

- you **tokenised** text and turned words into numbers,
- you **trained** a model to learn which words follow which,
- and you made it **generate** brand-new text one word at a time.

The difference between this and ChatGPT is mostly **scale**: more data, more context, and far more
computing power — but the central trick, *predicting the next word*, is exactly what you just did.